In [ ]:
%load_ext autoreload
%autoreload 2


# SPICE → GDS Layout Generator
SSCS Chipathon 2026 — gLayout Track

Converts SPICE subcircuit netlist to DRC-clean GDSII layout.
Includes AC simulation, DRC/LVS/PEX verification.

**PDK**: gf180mcuD  |  **Supply**: 1.8 V

In [ ]:
import sys, os
from pathlib import Path
root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180
from gdsfactory import Component
import gdstk, IPython.display

from core import (
    spice_to_gds, display_component,
    run_comparator_tran, run_comparator_pvt,
    run_drc, run_lvs, run_pex,
    compare_comp_pre_post,
)

In [ ]:
# ==========================================
# 1. Define SPICE Netlist
# ==========================================
# Edit this cell to change your circuit.
# Models: nfet_03v3 (NMOS), pfet_03v3 (PMOS)
# Supply: 1.8V, body connections: NMOS→vss, PMOS→vdd

netlist = """
.lib "/home/huda/.volare/gf180mcuD/libs.tech/ngspice/sm141064.ngspice" typical
.subckt comp_strongarm vin_p vin_n clk vout_p vout_n vdd vss
Mtail  n1 clk vss vss nfet_03v3 W=5u L=0.5u
M2 n2 vin_p n1 vss nfet_03v3 W=8u L=0.5u
M3 n3 vin_n n1 vss nfet_03v3 W=8u L=0.5u
M4 n2 n3 vdd vdd pfet_03v3 W=4u L=0.5u
M5 n3 n2 vdd vdd pfet_03v3 W=4u L=0.5u
M6 n2 n3 n1 vss nfet_03v3 W=3u L=0.5u
M7 n3 n2 n1 vss nfet_03v3 W=3u L=0.5u
M8 n2 clk vdd vdd pfet_03v3 W=2u L=0.5u
M9 n3 clk vdd vdd pfet_03v3 W=2u L=0.5u
M10 vout_p n2 vdd vdd pfet_03v3 W=4u L=0.5u
M11 vout_p n2 vss vss nfet_03v3 W=2u L=0.5u
M12 vout_n n3 vdd vdd pfet_03v3 W=4u L=0.5u
M13 vout_n n3 vss vss nfet_03v3 W=2u L=0.5u
.ends
"""

In [ ]:
# ==========================================
# 3. Pre-Simulation (Transient + PVT)
# ==========================================

print("--- Delay Propagation ---")
pre = run_comparator_tran(netlist, "comp_strongarm",
                           vdd=1.8, vcm=0.9,
                           clk_period=10e-9, tstop=30e-9)

def _fmt_v(v, unit=""):
    return f"{v:.3g} {unit}" if v is not None else "N/A"

print(f"  t_delay    = {_fmt_v(pre.get('tdelay'), 's')}")
print(f"  Vout_high  = {_fmt_v(pre.get('vout_high'), 'V')}")
print(f"  Vout_low   = {_fmt_v(pre.get('vout_low'), 'V')}")

print("\n--- Offset Measurement ---")
off = run_comparator_tran(netlist, "comp_strongarm",
                           vdd=1.8, vcm=0.9,
                           clk_period=20e-9, tstop=100e-9,
                           measure_offset=True)
print(f"  V_os       = {_fmt_v(off.get('vos'), 'V')}")

print("\n--- PVT Corners ---")
pvt = run_comparator_pvt(netlist, "comp_strongarm",
                          vdd=1.8, vcm=0.9,
                          clk_period=10e-9, tstop=30e-9)

print("\nPVT Results:")
print(f"{'Corner':<25} {'Temp':>6} {'VDD':>7} {'t_delay':>12}")
print("-" * 52)
for c in pvt["corners"]:
    td = _fmt_v(c.get('tdelay'), 's')
    print(f"{c['description']:<25} {c['temperature']:>5}C {c['vdd']:>6}V {td:>12}")

s = pvt["summary"]
print(f"\nSummary: tdelay_typ={_fmt_v(s.get('tdelay_typ'), 's')}")
print(f"         tdelay_min={_fmt_v(s.get('tdelay_min'), 's')}")
print(f"         tdelay_max={_fmt_v(s.get('tdelay_max'), 's')}")
print(f"         vos       ={_fmt_v(s.get('vos'), 'V')}")

In [ ]:
# ==========================================
# 2. Generate GDS Layout
# ==========================================

GDS_PATH = os.path.join(os.getcwd(), "out.gds")

result = spice_to_gds(netlist, mode="analog", add_labels=True)
result.write_gds(GDS_PATH)
print(f"Layout written to {GDS_PATH}")
display_component(result, scale=0.5)

In [ ]:
# ==========================================
# 4. DRC / LVS / PEX (optional)
# ==========================================
# Requires Magic, netgen, and PDK installed.

os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/.volare"))
os.environ.setdefault("PDK", "gf180mcuD")
os.environ.setdefault("PDKPATH", os.path.join(os.environ["PDK_ROOT"], os.environ["PDK"]))
os.environ.setdefault("STD_CELL_LIBRARY", "gf180mcu_fd_sc_mcu7t5v0")

# Run layout with verification (generates DRC+LVS+PEX automatically):
# result = spice_to_gds(netlist, mode="analog", add_labels=True, run_checks=True)

# Or run individually:
# drc = run_drc("out.gds", cell_name="comp_strongarm")
# lvs = run_lvs("out.gds", netlist_content=netlist, cell_name="comp_strongarm")
# pex = run_pex("out.gds", cell_name="comp_strongarm", mode=2)
#
# # Compare pre-layout vs post-layout (PEX):
# cmp = compare_comp_pre_post(netlist, pex["pex_path"], "comp_strongarm",
#                               vdd=1.8, vcm=0.9)
# print(f"PRE: tdelay={cmp['pre']['tdelay']} vos={cmp['pre']['vos']}")
# print(f"POST: tdelay={cmp['post']['tdelay']} vos={cmp['post']['vos']}")